<a href="https://colab.research.google.com/github/sarmanoveduard-design/intelligent-knowledge-base/blob/main/generation-benchmark/notebooks/generation_benchmark_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MOST — Generation Benchmark Colab

Один и тот же frozen Top‑10 + один prompt для трёх моделей: `qwen3:8b`, `gpt-5.6-luna`, `gpt-5.6-terra`.

- Qwen запускается локально в Colab через Ollama с `think=false`.
- Luna/Terra запускаются через OpenAI Responses API с `reasoning.effort=none`.
- Приватный TEAM-контекст и API-ключи в notebook/GitHub не хранятся.
- Для точного benchmark загрузите `model_input_top10.json` и `generation_v1_benchmark.txt`.
- Для Qwen включите GPU: Runtime → Change runtime type → T4 GPU.


In [ ]:
!pip -q install "openai>=2.0,<3" requests pandas


In [ ]:
#@title Настройки
MODELS_TO_RUN = "all3" #@param ["qwen3:8b", "gpt-5.6-luna", "gpt-5.6-terra", "all3"]
RUN_SCOPE = "one" #@param ["one", "all"]
QUERY_NUMBER = 87 #@param {type:"integer"}
CONFIRM_FULL_RUN = False #@param {type:"boolean"}
STRICT_BENCHMARK = True #@param {type:"boolean"}

EXPECTED_INPUT_SHA256 = "1b94bfca536b36e386d663572f72e9abe91b19da725d83b18df040277fd52f93"
EXPECTED_PROMPT_SHA256 = "f05cb6f1ec4e05d20aeb1cf7bfc8c9d2e508450b42073698707707a6de923a6c"

if RUN_SCOPE == "all" and not CONFIRM_FULL_RUN:
    raise RuntimeError("Для полного прогона поставьте CONFIRM_FULL_RUN=True")

SELECTED_MODELS = ["qwen3:8b", "gpt-5.6-luna", "gpt-5.6-terra"] if MODELS_TO_RUN == "all3" else [MODELS_TO_RUN]
print("MODELS=", SELECTED_MODELS)
print("RUN_SCOPE=", RUN_SCOPE)


In [ ]:
#@title Загрузите frozen benchmark input + prompt
from google.colab import files
from pathlib import Path
import hashlib, json

uploaded = files.upload()
required = {"model_input_top10.json", "generation_v1_benchmark.txt"}
missing = required - set(uploaded)
if missing:
    raise RuntimeError(f"Не загружены файлы: {sorted(missing)}")

INPUT_PATH = Path("/content/model_input_top10.json")
PROMPT_PATH = Path("/content/generation_v1_benchmark.txt")

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

input_hash = sha256_file(INPUT_PATH)
prompt_hash = sha256_file(PROMPT_PATH)
dataset = json.loads(INPUT_PATH.read_text(encoding="utf-8-sig"))
prompt = PROMPT_PATH.read_text(encoding="utf-8-sig").strip()

if len(dataset.get("queries", [])) != 119:
    raise RuntimeError("Ожидалось 119 вопросов")
if dataset.get("context_top_k") != 10:
    raise RuntimeError("Ожидался context_top_k=10")
for case in dataset["queries"]:
    if not str(case.get("query", "")).strip():
        raise RuntimeError(f"Пустой вопрос: {case.get('query_number')}")
    if len(case.get("contexts", [])) != 10:
        raise RuntimeError(f"Не 10 контекстов: {case.get('query_number')}")
    if "answer_hint" in json.dumps(case, ensure_ascii=False):
        raise RuntimeError("STOP: answer_hint leaked into model input")

if STRICT_BENCHMARK:
    if input_hash != EXPECTED_INPUT_SHA256:
        raise RuntimeError(f"MODEL INPUT SHA mismatch: {input_hash}")
    if prompt_hash != EXPECTED_PROMPT_SHA256:
        raise RuntimeError(f"PROMPT SHA mismatch: {prompt_hash}")

print("BENCHMARK INPUT: OK")
print("MODEL_INPUT_SHA256=", input_hash)
print("PROMPT_SHA256=", prompt_hash)


In [ ]:
#@title Подготовить OpenAI key и Qwen/Ollama
import getpass, os, requests, shutil, subprocess, time

needs_openai = any(m.startswith("gpt-") for m in SELECTED_MODELS)
if needs_openai and not os.environ.get("OPENAI_API_KEY"):
    key = None
    try:
        from google.colab import userdata
        key = userdata.get("OPENAI_API_KEY")
    except Exception:
        pass
    if not key:
        key = getpass.getpass("OPENAI_API_KEY: " ).strip()
    if not key:
        raise RuntimeError("OPENAI_API_KEY не задан")
    os.environ["OPENAI_API_KEY"] = key

if "qwen3:8b" in SELECTED_MODELS:
    if shutil.which("nvidia-smi"):
        subprocess.run(["nvidia-smi"], check=False)
    else:
        print("WARNING: GPU не обнаружен, Qwen будет очень медленным")
    if not shutil.which("ollama"):
        subprocess.run(["bash", "-lc", "curl -fsSL https://ollama.com/install.sh | sh"], check=True)
    def ready():
        try:
            return requests.get("http://127.0.0.1:11434/api/tags", timeout=2).ok
        except Exception:
            return False
    if not ready():
        log = open("/tmp/ollama-colab.log", "w")
        subprocess.Popen(["ollama", "serve"], stdout=log, stderr=subprocess.STDOUT, env=os.environ.copy())
        for _ in range(60):
            if ready():
                break
            time.sleep(1)
        else:
            raise RuntimeError("Ollama не запустился")
    subprocess.run(["ollama", "pull", "qwen3:8b"], check=True)

print("ENVIRONMENT: READY")


In [ ]:
#@title Общий runner
import json, math, re, time
from pathlib import Path
import pandas as pd, requests
from openai import OpenAI

def safe_name(s):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", s)

def pct(values, f):
    if not values: return None
    a = sorted(values)
    return a[max(0, math.ceil(len(a) * f) - 1)]

def user_message(case):
    p = ["ВОПРОС ПОЛЬЗОВАТЕЛЯ:", case["query"], "", "ПРЕДОСТАВЛЕННЫЙ КОНТЕКСТ:"]
    for ctx in case["contexts"]:
        p += ["", f"[Источник {ctx['rank']}]", f"document_id: {ctx['document_id']}", f"section_ref: {ctx['section_ref']}", "text:", ctx["text"]]
    return "\n".join(p)

client = OpenAI() if any(m.startswith("gpt-") for m in SELECTED_MODELS) else None

def call_model(model, case):
    started = time.perf_counter()
    if model == "qwen3:8b":
        r = requests.post("http://127.0.0.1:11434/api/chat", json={
            "model": model,
            "messages": [{"role": "system", "content": prompt}, {"role": "user", "content": user_message(case)}],
            "think": False, "stream": False, "keep_alive": "30m",
            "options": {"num_ctx": 8192, "num_predict": 512},
        }, timeout=600)
        r.raise_for_status()
        raw = r.json()
        answer = (raw.get("message") or {}).get("content", "").strip()
        return {
            "answer": answer, "latency_seconds": time.perf_counter() - started,
            "input_tokens": raw.get("prompt_eval_count"), "output_tokens": raw.get("eval_count"),
            "total_tokens": (raw.get("prompt_eval_count") or 0) + (raw.get("eval_count") or 0),
        }
    resp = client.responses.create(model=model, reasoning={"effort": "none"}, instructions=prompt, input=user_message(case), max_output_tokens=512, store=False)
    u = resp.usage
    return {
        "answer": resp.output_text.strip(), "latency_seconds": time.perf_counter() - started,
        "input_tokens": u.input_tokens if u else None, "output_tokens": u.output_tokens if u else None,
        "total_tokens": u.total_tokens if u else None,
    }

def run_model(model, cases):
    out_dir = Path("/content/generation_results"); out_dir.mkdir(exist_ok=True)
    rows = []
    path = out_dir / f"{safe_name(model)}.jsonl"
    with path.open("w", encoding="utf-8") as f:
        for i, case in enumerate(cases, 1):
            qn = int(case["query_number"])
            try:
                x = call_model(model, case)
                row = {"status": "ok", "model": model, "query_number": qn, "query_id": case.get("query_id"), "prompt_sha256": prompt_hash, "model_input_sha256": input_hash, **x}
                print(f"[{model}] [{i:03d}/{len(cases):03d}] q={qn:03d} OK {x['latency_seconds']:.2f}s")
            except Exception as e:
                row = {"status": "error", "model": model, "query_number": qn, "query_id": case.get("query_id"), "error_type": type(e).__name__, "error_message": str(e)[:300]}
                print(f"[{model}] [{i:03d}/{len(cases):03d}] q={qn:03d} ERROR {type(e).__name__}")
            rows.append(row); f.write(json.dumps(row, ensure_ascii=False) + "\n"); f.flush()
    ok = [r for r in rows if r["status"] == "ok"]
    lat = [float(r["latency_seconds"]) for r in ok]
    summary = {
        "model": model, "selected_query_count": len(cases), "successful_query_count": len(ok), "error_count": len(rows) - len(ok),
        "p50_latency_seconds": pct(lat, .5), "p95_latency_seconds": pct(lat, .95),
        "total_input_tokens": sum(int(r.get("input_tokens") or 0) for r in ok),
        "total_output_tokens": sum(int(r.get("output_tokens") or 0) for r in ok),
        "prompt_sha256": prompt_hash, "model_input_sha256": input_hash,
    }
    (out_dir / f"{safe_name(model)}_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    return rows, summary


In [ ]:
#@title Запуск и скачивание результатов
if RUN_SCOPE == "one":
    cases = [q for q in dataset["queries"] if int(q["query_number"]) == int(QUERY_NUMBER)]
    if len(cases) != 1:
        raise RuntimeError("query_number не найден ровно один раз")
else:
    cases = list(dataset["queries"])

all_summaries = []
for model in SELECTED_MODELS:
    print("\n===", model, "===")
    _, summary = run_model(model, cases)
    all_summaries.append(summary)
display(pd.DataFrame(all_summaries))

import shutil
from google.colab import files
zip_path = shutil.make_archive("/content/MOST_generation_benchmark_results", "zip", "/content/generation_results")
print("RESULTS=", zip_path)
files.download(zip_path)
